# 原 Stage1 2026Q3 两个最大选择损失区间：独立日线核价

复用冻结账户损失日期，只做 Yahoo SOXL/SOXX 日线方向与事件跨来源核对；不同价格时钟不强求相等。

In [1]:
import os,json,hashlib,tarfile,tempfile,subprocess,shlex
from pathlib import Path
from zoneinfo import ZoneInfo
from datetime import datetime
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd();card=root/'research/stage1_original_0bp_july_quote_crosscheck_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['selection_identity'];snap=root/'data/snapshots/stage1_original_0bp_july_quote_crosscheck_20261023';sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('new_model_fits','new_native_accounts','SOXX_backtest_replays'))
prior=root/src['source_card'];assert sha(prior/'summary.json')==src['source_summary_sha256'] and sha(prior/'result.json')==src['source_result_sha256']
prior_summary=json.loads((prior/'summary.json').read_text());assert prior_summary['recorder_id']==src['source_recorder_id'] and prior_summary['nas_readback']['sha256']==src['source_NAS_sha256']
transport=json.loads((card/'transport_result.json').read_text());receipt=json.loads((card/'nas_source_archive.json').read_text());assert len(transport['attempts'])==2 and all(x['status']==200 for x in transport['attempts']) and receipt['verified']
frames={};events={}
for q in transport['attempts']:
 sym=q['symbol'];path=snap/(sym+'.json');assert sha(path)==q['sha256'] and path.stat().st_size==q['size_bytes']
 body=json.loads(path.read_text());assert body['chart']['error'] is None and len(body['chart']['result'])==1
 item=body['chart']['result'][0];assert item['meta']['symbol']==sym and item['meta']['exchangeTimezoneName']=='America/New_York'
 quote=item['indicators']['quote'][0];dates=[pd.Timestamp(datetime.fromtimestamp(t,ZoneInfo('America/New_York')).date()) for t in item['timestamp']]
 f=pd.DataFrame({'date':dates,'instrument':sym,'open':quote['open'],'high':quote['high'],'low':quote['low'],'close':quote['close']});assert len(f)==20 and f.date.is_unique and f.date.is_monotonic_increasing and f[['open','high','low','close']].notna().all().all() and (f[['open','high','low','close']]>0).all().all()
 assert (f.high>=f[['open','close','low']].max(axis=1)).all() and (f.low<=f[['open','close','high']].min(axis=1)).all()
 frames[sym]=f.set_index('date');events[sym]=item.get('events',{})
assert frames['SOXL'].index.equals(frames['SOXX'].index)
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri='data/qlib/minute_information_pit_allocator_20261012/daily_prices',region='us',kernels=1)
cal=pd.DatetimeIndex(D.calendar(start_time='2026-07-01',end_time='2026-07-29',freq='day'));assert len(cal)==20 and cal.equals(frames['SOXL'].index)
with R.uri_context(prior_summary['recorder_uri']):saved=R.get_recorder(recorder_id=src['source_recorder_id'],experiment_id=src['source_experiment_id']).load_object('analysis/daily')
assert len(saved)==944
comparison=[]
for start,end in zip(src['two_worst_Q3_negative_trade_dates'],src['matched_exit_dates']):
 st=pd.Timestamp(start);en=pd.Timestamp(end);row=saved.loc[pd.to_datetime(saved.trade_date).eq(st)];assert len(row)==1 and pd.Timestamp(row.exit_date.iloc[0])==en
 price={}
 for sym in ('SOXL','SOXX'):
  f=frames[sym];assert st in f.index and en in f.index
  price[sym]={'Yahoo_open_to_next_open':float(f.loc[en,'open']/f.loc[st,'open']-1),'Yahoo_close_to_next_close':float(f.loc[en,'close']/f.loc[st,'close']-1),'Yahoo_trade_open':float(f.loc[st,'open']),'Yahoo_exit_open':float(f.loc[en,'open']),'Yahoo_trade_close':float(f.loc[st,'close']),'Yahoo_exit_close':float(f.loc[en,'close'])}
 open_spread=price['SOXL']['Yahoo_open_to_next_open']-price['SOXX']['Yahoo_open_to_next_open'];close_spread=price['SOXL']['Yahoo_close_to_next_close']-price['SOXX']['Yahoo_close_to_next_close'];stage1=float(row.raw_relative_forward_price_return.iloc[0]);assert open_spread<0 and close_spread<0 and stage1<0
 comparison.append({'trade_date':start,'exit_date':end,'Stage1_0947_relative_forward_simple_return':stage1,'Stage1_physical_selection_log':float(row.SOXL_selection_log.iloc[0]),'Yahoo_daily_open_to_open_relative_spread':open_spread,'Yahoo_daily_close_to_close_relative_spread':close_spread,'by_instrument':price,'same_clocks':False,'same_price_adjustment_vintage':False})
rows=pd.concat(frames.values()).reset_index().sort_values(['date','instrument']);static=pd.concat({'feature':rows.set_index(['date','instrument'])[['open','high','low','close']]},axis=1);static.index.names=['datetime','instrument'];ds=DatasetH(handler=DataHandlerLP.from_df(static),segments={'July':('2026-07-01','2026-07-29')});assert len(ds.prepare('July',col_set=['feature']))==40
report={'decision':'two_largest_Q3_Stage1_loss_intervals_corrobated_by_independent_Yahoo_daily_directions_not_exact_0947_prices','July_Yahoo_session_days':20,'Yahoo_split_or_dividend_event_counts':{k:{event:len(value) for event,value in v.items()} for k,v in events.items()},'comparison':comparison,'all_four_Yahoo_window_relative_spreads_negative':True,'account_source_days_hindsight_selected_for_quality_only':True,'Yahoo_current_revision_not_historical_PIT':True,'no_exact_0947_independent_vendor_quote':True,'model_fits':0,'native_accounts':0}
tracking=card/'qlib_recorder';assert not tracking.exists()
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 rec=R.get_recorder();rid=rec.id;eid=str(rec.experiment_id);ds.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,dataset=ds,raw_source_hashes=transport['attempts']);rec.save_objects(artifact_path='analysis',Yahoo_daily=rows,report=report);rec.log_metrics(Yahoo_July_sessions=20,matched_loss_intervals=2,model_fits=0,native_accounts=0)
summary={'experiment':plan['experiment'],'decision':report['decision'],'objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'source_archive_sha256':receipt['sha256'],'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'result':'result.json','new_model_fits':0,'new_native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Independent Yahoo daily OHLC windows differ from frozen Stage1 09:47 marks and are current revised history.','Selected loss dates are hindsight source-quality checks, not factor or trading evidence.','Corporate-action event absence in Yahoo JSON does not prove all issuer action handling.']}
(card/'result.json').write_text(json.dumps(report,indent=2)+'\n');(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n');e=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
with tempfile.TemporaryDirectory(prefix='july-quote-recorder-') as t:
 archive=Path(t)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(e['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==e['sha256']
 with tarfile.open(archive,'r:gz') as z:assert any(m.name.endswith('/artifacts/analysis/Yahoo_daily') for m in z.getmembers())
summary['nas_evidence']=e;summary['nas_readback']={'verified':True,'sha256':e['sha256']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');print([(x['trade_date'],x['Stage1_0947_relative_forward_simple_return'],x['Yahoo_daily_open_to_open_relative_spread'],x['Yahoo_daily_close_to_close_relative_spread']) for x in comparison])


2026/10/23 07:22:56 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[16920:MainThread](2026-10-23 07:22:57,713) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[16920:MainThread](2026-10-23 07:22:57,715) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[16920:MainThread](2026-10-23 07:22:57,715) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


...

[16920:MainThread](2026-10-23 07:22:57,753) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_original_0bp_july_quote_crosscheck_20261023.


[16920:MainThread](2026-10-23 07:22:57,754) INFO - qlib.workflow - [exp.py:258] - Experiment 524327802807416584 starts running ...


[16920:MainThread](2026-10-23 07:22:57,836) INFO - qlib.workflow - [recorder.py:345] - Recorder 3617574810dd4a438ec80f25d89359dc starts running under Experiment 524327802807416584 ...


[16920:MainThread](2026-10-23 07:22:58,875) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


[('2026-07-06', -0.1203560953211904, -0.10776331836834563, -0.09960588639564849), ('2026-07-27', -0.14759921384769237, -0.12812058640513935, -0.0972379279764739)]


# 原 Stage1 2026Q3 两个最大选择损失区间：独立日线核价

## 中文结论

冻结的原 Stage1 A0、完整 $100,000 SOXL/SOXX/CASH、0bp 附加滑点与 IBKR Tiered 费用账户中，2026Q3 的 SOXL 选择项为 −0.143901 对数点。7 月 6 日和 7 月 27 日两个区间贡献了该季度负选择损失绝对值的 57.95%。这两天是事后按损失选出的**数据核查样本**，不能据此制定交易规则。

本次按冻结计划只取 Yahoo 当前历史日线各一份 SOXL、SOXX 数据，并用原生 Qlib 日历、DatasetH 和 Recorder 核对。7 月 6→7 日，Stage1 已保存的 09:47→次日 09:47 SOXL−SOXX 简单收益差为 −12.04 个百分点；Yahoo 开盘到次日开盘为 −10.78 个百分点，收盘到次日收盘为 −9.96 个百分点。7 月 27→28 日相应为 −14.76、−12.81、−9.72 个百分点。四个 Yahoo 日线差值均为负，且 SOXL 跌幅显著大于 SOXX。独立日线支持这两段损失的**方向和大致量级**，没有显示出明显的单一 Stage1 价格源异常。

这不是精确复算：Yahoo 日线开收盘与 Stage1 09:47 时钟不同，而且是当前修订历史；不能由此认证 09:47 精确报价、实盘账户费用净损失、历史当时可得性。Yahoo 响应中没有拆分或分红事件对象，仅说明这份响应没有报告事件，不证明所有公司行动处理正确。原账户物理选择对数损失分别为 −0.03904、−0.04726，仍以冻结 Stage1 Recorder 为准。

本卡零模型拟合、零新账户、零交易规则。独立复核确认计划、原始响应哈希、Qlib 40 行读回及 NAS 原始源和 Recorder 归档读回一致。**推广决定：不推广；目标仍未达成。** 下一步应继续在完整 Stage1 的 0bp 账户上，对最差区间和较好区间比较可提前观察的状态，提出事先冻结、可证伪且不同于已失败尾部概率路由的新机制；不得用这两个事后日期调阈值或选择窗口。
